<a href="https://colab.research.google.com/github/matheusj4sus/pipeline-ml-body-fat/blob/main/Atividade_Integradora_Q2_Classificacao_Gordura_Corporal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Integradora — Pipeline Completo de Machine Learning
## Questão 2 · Classificação: identificando gordura corporal elevada
### Tema: academia / avaliação física

**Disciplina:** Tópicos Avançados em Sistemas de Informação II — UNICID  
**Professor:** Prof. Me. Gabriel Rosa Paz  
**Alunos:** Matheus De Jesus Araujo e Daniel Vieira  
**Ambiente:** Google Colab · Python 3 · pandas · seaborn · scikit-learn

> Esta é a **Parte 2** da atividade (classificação). A Parte 1 (regressão) usa a **mesma base de dados** (Body Fat) para **prever o valor** do % de gordura. Aqui, o problema é outro: **classificar** cada pessoa em uma de duas categorias, a partir do mesmo % de gordura.

**Como executar:** menu *Ambiente de execução → Reiniciar e executar tudo*. O notebook baixa os dados sozinho; se a internet do Colab falhar, ele pede o upload do arquivo.

### Sumário (segue a numeração do enunciado)
1. Escolha do dataset
2. Explicação do problema e do objetivo
3. Importação do dataset e separação de X e y
4. Análise exploratória (EDA)
5. Limpeza e pré-processamento
6. Gráficos e interpretação
7. Divisão em treino e teste
8. Algoritmos e hiperparâmetros
9. Necessidade de normalização
10. Comparação por métricas
11. Conclusão final

## 1) Escolha do dataset

| Item | Informação |
|---|---|
| **Nome** | *Fitting Percentage of Body Fat to Simple Body Measurements* (**Body Fat**) |
| **Fonte original (documentação)** | Journal of Statistics Education (JSE) — Data Archive: https://jse.amstat.org/datasets/fat.txt |
| **Arquivo de dados** | https://jse.amstat.org/datasets/fat.dat.txt |
| **Artigo de referência** | Johnson, R. W. (1996). *Fitting Percentage of Body Fat to Simple Body Measurements*. Journal of Statistics Education, 4(1). |
| **Origem dos dados** | Dr. A. Garth Fisher, Human Performance Research Center, Brigham Young University (EUA); distribuição livre para uso não comercial. Estudo original: Penrose, Nelson & Fisher (1985). |
| **Tamanho** | 252 homens · 19 variáveis originais |
| **Tipo de problema** | **Classificação binária** (target categórico criado a partir do % de gordura) |

**Por que é um problema de classificação?** Aqui o target **não é o número** do % de gordura. É uma **classe**: `1` = gordura corporal **elevada** (≥ 25 %) e `0` = **abaixo** de 25 %. A resposta que o modelo dá é "sim/não", e não um valor contínuo.

**De onde vem o limiar de 25 %?** É uma **decisão de domínio** (não aprendida dos dados): 25 % de gordura corporal é o valor usado em tabelas de classificação para homens, como a do *American Council on Exercise* (ACE), para a faixa de obesidade. Como é uma regra **fixa e definida antes de olhar o desempenho dos modelos**, ela **não causa vazamento de dados**.

**Mesma base da Parte 1.** A fonte é a mesma, mas o problema muda: regressão (valor) na Parte 1, classificação (categoria) na Parte 2. Os dados foram **reimportados e tratados de novo neste notebook**, para que ele rode sozinho.

**Dataset não utilizado nas aulas:** este conjunto não foi usado nos exemplos da disciplina.

## 2) Explicação do problema e do objetivo

**Domínio.** Avaliação física em academias. Muitas vezes o instrutor não precisa do número exato, só quer saber **se o aluno está numa faixa de gordura elevada** para orientar o treino e a dieta.

| Pergunta | Resposta |
|---|---|
| **O que representa cada amostra?** | Um homem adulto avaliado fisicamente. |
| **Quais são as features?** | Idade, peso, altura e 10 circunferências corporais (pescoço, peito, abdômen, quadril, coxa, joelho, tornozelo, bíceps, antebraço e punho). |
| **Qual é a classe prevista?** | `gordura_elevada`: **1** se % de gordura (Siri) ≥ 25 %, **0** caso contrário. |
| **Binária ou multiclasse?** | **Binária.** A **classe positiva (1)** é a de gordura elevada. |
| **Qual é a fonte?** | JSE Data Archive (links na seção 1). |

**Objetivo do modelo.** Identificar, usando só **balança e fita métrica**, quem tem gordura corporal elevada, **sem precisar da pesagem subaquática**.

> ⚠️ **Uso educacional.** Este trabalho é um exercício de Machine Learning. O modelo **não** é uma ferramenta de diagnóstico clínico ou nutricional.

### Colunas que **não** entram em X (evitar *target leakage*)

Como o target vem do % de gordura, qualquer coluna calculada **a partir dele** entregaria a resposta:

| Coluna | Por que sai de X |
|---|---|
| `gordura_siri` | É a base do target (o target é `gordura_siri ≥ 25`). |
| `gordura_brozek` | Mesma informação, calculada por outra fórmula (quase uma cópia). |
| `densidade` | O % de gordura é calculado **diretamente** dela. |
| `massa_magra_lb` | Calculada **usando** o % de gordura. |
| `indice_adiposidade` | Função de peso e altura (já estão em X): redundante. |
| `caso` | Só identificador; vira o índice. |

In [ ]:
import io
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, make_scorer)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 3) Importação do dataset e separação de X e y

O arquivo original **não tem cabeçalho**: os nomes das colunas seguem a ordem da documentação do JSE. Depois de carregar, separamos:

- `X`: as features (medidas simples);
- `y_pct`: o % de gordura **contínuo**, usado **apenas** para criar a classe e para análises auxiliares (nunca como feature);
- `y`: o target de classificação (`gordura_elevada`), criado com uma regra fixa: `y = 1` se `y_pct ≥ 25`.

In [ ]:
COLUNAS = [
    "caso", "gordura_brozek", "gordura_siri", "densidade", "idade",
    "peso_lb", "altura_pol", "indice_adiposidade", "massa_magra_lb",
    "pescoco", "peito", "abdomen", "quadril", "coxa", "joelho",
    "tornozelo", "biceps", "antebraco", "punho",
]
FONTES = [
    "https://jse.amstat.org/datasets/fat.dat.txt",
    "http://jse.amstat.org/datasets/fat.dat.txt",
]


def ler_fonte(fonte):
    """Lê o arquivo (URL ou caminho local) e devolve um DataFrame numérico."""
    if fonte.startswith("http"):
        resposta = requests.get(fonte, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resposta.raise_for_status()
        conteudo = io.StringIO(resposta.text)
    else:
        conteudo = fonte
    dados = pd.read_csv(conteudo, sep=r"\s+", header=None, names=COLUNAS)
    return dados.apply(pd.to_numeric, errors="raise")


def carregar_dados():
    for fonte in FONTES:
        try:
            dados = ler_fonte(fonte)
            if dados.shape == (252, len(COLUNAS)):
                return dados, fonte
            print(f"Formato inesperado em {fonte}: {dados.shape}")
        except Exception as erro:
            print(f"Falha ao ler {fonte}: {erro}")
    # Plano B: upload manual do arquivo fat.dat.txt (link na seção 1)
    from google.colab import files
    print("Baixe o arquivo fat.dat.txt pelo link da seção 1 e envie-o abaixo:")
    nome = next(iter(files.upload()))
    return ler_fonte(nome), nome


df_bruto, fonte_usada = carregar_dados()
df_bruto = df_bruto.set_index("caso")
print(f"Fonte utilizada: {fonte_usada}")
print(f"Formato do dataset original: {df_bruto.shape}")
df_bruto.head()

Fonte utilizada: https://jse.amstat.org/datasets/fat.dat.txt
Formato do dataset original: (252, 18)


,gordura_brozek,gordura_siri,densidade,idade,peso_lb,altura_pol,indice_adiposidade,massa_magra_lb,pescoco,peito,abdomen,quadril,coxa,joelho,tornozelo,biceps,antebraco,punho
caso,,,,,,,,,,,,,,,,,,
1,12.6,12.3,1.0708,23,154.25,67.75,23.7,134.9,36.2,93.1,85.2,94.5,59.0,37.3,21.9,32.0,27.4,17.1
2,6.9,6.1,1.0853,22,173.25,72.25,23.4,161.3,38.5,93.6,83.0,98.7,58.7,37.3,23.4,30.5,28.9,18.2
3,24.6,25.3,1.0414,22,154.00,66.25,24.7,116.0,34.0,95.8,87.9,99.2,59.6,38.9,24.0,28.8,25.2,16.6
4,10.9,10.4,1.0751,26,184.75,72.25,24.9,164.7,37.4,101.8,86.4,101.2,60.1,37.3,22.8,32.4,29.4,18.2
5,27.8,28.7,1.0340,24,184.25,71.25,25.6,133.1,34.4,97.3,100.0,101.9,63.2,42.2,24.0,32.2,27.7,17.7


In [ ]:
TARGET_PCT = "gordura_siri"
LIMIAR = 25.0                       # % de gordura a partir do qual a classe é 1 (decisão de domínio)
COLS_VAZAMENTO = ["gordura_brozek", "densidade", "massa_magra_lb"]
COLS_REDUNDANTES = ["indice_adiposidade"]

y_pct = df_bruto[TARGET_PCT].copy()                       # % de gordura (auxiliar, NÃO é feature)
X = df_bruto.drop(columns=[TARGET_PCT] + COLS_VAZAMENTO + COLS_REDUNDANTES).copy()

# Target de classificação: regra fixa, sem aprender nada dos dados
y = (y_pct >= LIMIAR).astype(int).rename("gordura_elevada")

ROTULOS = {0: "Abaixo de 25% (0)", 1: "Gordura elevada, ≥ 25% (1)"}
ROTULOS_CURTOS = {0: "< 25%", 1: "≥ 25%"}

print("Features (X):", list(X.columns))
print("\nPrimeiras linhas de X:")
display(X.head())
print("Primeiros valores de y (classe) ao lado do % de gordura que a originou:")
display(pd.DataFrame({"gordura_siri_%": y_pct, "gordura_elevada": y}).head(8))

## 4) Análise exploratória (EDA)

Antes de qualquer transformação, vamos entender **estrutura, tipos, escala, qualidade e balanceamento das classes**.

In [ ]:
print("Dimensão de X (linhas, colunas):", X.shape)
print("\nTipos de dados e não-nulos:")
X.info()
print("\nEstatísticas descritivas de X:")
display(X.describe(include="all").T)
print("\nValores nulos por coluna:")
print(X.isnull().sum())
print("\nLinhas duplicadas em X:", X.duplicated().sum())
print("\nDistribuição das classes (contagem):")
print(y.value_counts())
print("\nDistribuição percentual:")
print(y.value_counts(normalize=True).round(3))

In [ ]:
altura_min = X["altura_pol"].min()
print("DIAGNÓSTICO DA EDA")
print(f"• Amostras x features: {X.shape[0]} x {X.shape[1]}")
print(f"• Valores nulos (total em X): {int(X.isnull().sum().sum())} | nulos em y: {int(y.isnull().sum())}")
print(f"• Linhas duplicadas em X: {int(X.duplicated().sum())}")
print(f"• Altura mínima: {altura_min:.1f} pol = {altura_min * 2.54:.0f} cm  ->  "
      f"{'IMPLAUSÍVEL para um adulto' if altura_min < 48 else 'plausível'}")
print(f"• Registros com % de gordura <= 0: {int((y_pct <= 0).sum())}")
prop = y.mean()
print(f"• Proporção da classe positiva (≥ {LIMIAR:.0f}%): {prop:.1%}  |  classe majoritária: {1 - prop:.1%}")

### Interpretação da EDA

- **Tipos:** todas as features são numéricas; **não é necessário codificar categorias** (one-hot). A classe `y` já está em 0/1.
- **Nulos e duplicidades:** conforme a contagem acima, não há valores ausentes para imputar; se houver duplicidades, serão tratadas na seção 5.
- **Balanceamento (ponto central da classificação):** a classe positiva (gordura elevada) é **minoritária** (veja a proporção no diagnóstico). Isso muda como lemos as métricas: um modelo que **sempre responde "abaixo de 25%"** já acertaria a maioria dos casos, e mesmo assim **não encontraria ninguém** com gordura elevada. Por isso **acurácia sozinha engana**; olharemos **precisão, recall e F1** da classe positiva e compararemos com um *baseline*.
- **Escalas diferentes:** peso (kg) e abdômen (cm) têm valores muito maiores que punho e tornozelo, o que importa para KNN, SVC, Regressão Logística e SGD (seção 9).
- **Problemas de qualidade:** a documentação do JSE avisa de erros pontuais (uma altura impossível, % de gordura truncado em zero e incoerências entre as fórmulas Brozek e Siri). São tratados na seção 5.
- **Unidades:** peso em libras e altura em polegadas; converteremos para kg e cm.

## 5) Limpeza e pré-processamento

Cada decisão abaixo é **justificada**, sem aplicar nada "no automático":

| # | Decisão | Justificativa |
|---|---|---|
| 5.1 | **Corrigir a altura do caso 42** (29,5 → 69,5 pol) | Erro de digitação documentado pelos autores. Um adulto de ~90 kg com 75 cm é impossível. |
| 5.2 | **Remover linhas com target não confiável**: (a) % de gordura ≤ 0 e (b) Brozek e Siri incoerentes | Ambos indicam erro na medição/digitação da densidade. Como a classe depende desse valor, **a classe dessas linhas também seria duvidosa**. Prefiro perder poucas linhas a treinar com rótulo errado. |
| 5.3 | **Converter unidades** (lb → kg, pol → cm) | Conversão fixa, não aprendida dos dados: **sem data leakage**. |
| 5.4 | **Duplicidades** | Remover se existirem. |
| 5.5 | **Nulos** | Se existissem, usaria `SimpleImputer` **dentro do Pipeline** (ajustado só no treino). |
| 5.6 | **Outliers estatísticos: não remover** | Homens muito grandes ou com alto % de gordura são **reais e válidos**; removê-los eliminaria justamente a classe positiva que queremos detectar. |
| 5.7 | **Padronização (StandardScaler)** | **Não aplico agora.** Vai dentro de `Pipeline`, só nos modelos sensíveis à escala, ajustada **apenas no treino** (e, na validação cruzada, só no treino de cada fold). |
| 5.8 | **Criação da classe a partir do limiar de 25 %** | Regra fixa de domínio, definida antes da modelagem: não usa estatísticas do conjunto de teste. |

In [ ]:
# 5.1 Altura impossível ---------------------------------------------------------
mask_altura = X["altura_pol"] < 48
print("Registros com altura < 48 pol (≈ 1,22 m):")
display(X.loc[mask_altura, ["idade", "peso_lb", "altura_pol"]])
if 42 in X.index:
    X.loc[42, "altura_pol"] = 69.5          # correção documentada pelos autores
print("Altura mínima após a correção:", X["altura_pol"].min(), "pol")

# 5.2 Linhas com target não confiável ------------------------------------------
invalido_alvo = y_pct[y_pct <= 0].index
siri_esperado = 495 * (df_bruto["gordura_brozek"] + 414.2) / 457 - 450   # Siri implícito no Brozek
incoerente = (df_bruto["gordura_siri"] - siri_esperado).abs() > 1.0
incoerente = incoerente[incoerente].index
a_remover = invalido_alvo.union(incoerente)
print(f"\nTarget <= 0 (truncado na origem): {list(invalido_alvo)}")
print(f"Brozek x Siri incoerentes (|diferença| > 1 p.p.): {list(incoerente)}")
print(f"Total removido: {len(a_remover)} de {len(X)} linhas ({len(a_remover) / len(X):.1%})")
X, y, y_pct = X.drop(index=a_remover), y.drop(index=a_remover), y_pct.drop(index=a_remover)

In [ ]:
# 5.3 Conversão de unidades (regra fixa, sem aprender nada dos dados) -------------
X["peso_kg"] = X["peso_lb"] * 0.45359237
X["altura_cm"] = X["altura_pol"] * 2.54
X = X.drop(columns=["peso_lb", "altura_pol"])
X = X[["idade", "peso_kg", "altura_cm", "pescoco", "peito", "abdomen", "quadril",
       "coxa", "joelho", "tornozelo", "biceps", "antebraco", "punho"]]

# 5.4 Duplicidades -------------------------------------------------------------
mask_dup = X.duplicated()
print(f"Duplicidades encontradas: {int(mask_dup.sum())}")
if mask_dup.any():
    X, y, y_pct = X[~mask_dup], y[~mask_dup], y_pct[~mask_dup]

# 5.5 Nulos ------------------------------------------------------------------
print(f"Nulos restantes em X: {int(X.isnull().sum().sum())} | em y: {int(y.isnull().sum())}")

assert X.index.equals(y.index) and X.index.equals(y_pct.index)
print("\nFormato final de X e y:", X.shape, y.shape)
print("\nDistribuição final das classes:")
resumo = pd.DataFrame({"quantidade": y.value_counts().sort_index(),
                       "proporção": y.value_counts(normalize=True).sort_index().round(3)})
resumo.index = [ROTULOS[i] for i in resumo.index]
display(resumo)

## 6) Gráficos e interpretação

Cada gráfico responde a uma pergunta, com título, eixos identificados e leitura textual. (Aqui usamos o conjunto completo **só para exploração**; nenhuma transformação é aprendida a partir dele.)

In [ ]:
# Gráfico 1 — balanceamento das classes
contagem = y.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.countplot(x=y.map(ROTULOS), order=[ROTULOS[0], ROTULOS[1]], ax=ax, color="steelblue")
ax.bar_label(ax.containers[0], labels=[f"{v}  ({v / len(y):.0%})" for v in contagem.values])
ax.set(title="Distribuição das classes (gordura elevada × demais)",
       xlabel="Classe", ylabel="Quantidade de homens")
plt.tight_layout()
plt.show()
print(f"Classe majoritária = {contagem.max() / len(y):.1%} dos dados -> "
      f"um modelo que sempre prevê a maioria teria acurácia de {contagem.max() / len(y):.1%}.")

**Por que este gráfico?** Em classificação, o primeiro passo é conhecer o **balanceamento**. Ele define quais métricas fazem sentido.

**O que revela.** As classes **não são iguais**: a de gordura elevada é a minoritária. Isso tem duas consequências: (i) a **acurácia pode parecer boa mesmo com um modelo inútil**, porque basta acertar a classe maior; (ii) por isso usaremos **`stratify=y`** na divisão e avaliaremos por **F1, precisão e recall** da classe positiva.

In [ ]:
# Gráfico 2 — as 4 features mais associadas à classe, comparadas entre as classes
assoc = X.corrwith(y).abs().sort_values(ascending=False)    # exploração (conjunto completo)
top4 = list(assoc.index[:4])

fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
for ax, feat in zip(axes, top4):
    sns.boxplot(x=y.map(ROTULOS_CURTOS), y=X[feat], order=[ROTULOS_CURTOS[0], ROTULOS_CURTOS[1]],
                ax=ax, palette=["lightsteelblue", "salmon"], hue=y.map(ROTULOS_CURTOS), legend=False)
    ax.set(title=f"{feat} por classe", xlabel="% de gordura corporal", ylabel=f"{feat} (cm, kg ou anos)")
plt.tight_layout()
plt.show()
print("Associação (|correlação|) de cada feature com a classe:")
print(assoc.round(2).to_string())

**Por que este gráfico?** O boxplot compara a **distribuição de uma feature em cada classe**. Se as caixas ficam separadas, a feature ajuda a distinguir as classes.

**O que revela.** Nas medidas de tronco e volume (como abdômen e peso), a caixa do grupo "≥ 25 %" fica **mais alta** que a do grupo "< 25 %", com pouca sobreposição na mediana: essas medidas **separam bem** as classes. Já medidas como altura e punho tendem a se sobrepor muito, e por isso informam pouco. (Confirme com os valores impressos acima.) Como várias features são correlacionadas entre si, o modelo pode usar qualquer uma delas.

In [ ]:
# Gráfico 3 — duas features juntas: as classes se separam no plano abdômen x peso?
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(x=X["abdomen"], y=X["peso_kg"], hue=y.map(ROTULOS_CURTOS),
                hue_order=[ROTULOS_CURTOS[0], ROTULOS_CURTOS[1]],
                palette=["steelblue", "crimson"], alpha=0.75, s=45, ax=ax)
ax.set(title="Abdômen × peso, colorido pela classe de gordura",
       xlabel="Circunferência do abdômen (cm)", ylabel="Peso (kg)")
ax.legend(title="% de gordura")
plt.tight_layout()
plt.show()

**Por que este gráfico?** O scatter mostra se **duas features juntas** separam as classes, algo que o boxplot (uma feature por vez) não mostra.

**O que revela.** Os pontos vermelhos (gordura elevada) concentram-se no canto **superior direito** (abdômen e peso maiores), e os azuis, no canto inferior esquerdo. Existe uma **zona de transição** em que as classes se misturam. É nessa fronteira que os modelos terão mais dificuldade, o que será confirmado na análise dos erros (seção 10).

## 7) Divisão em treino e teste

- **80 % treino / 20 % teste**, com `random_state=42`.
- **Com `stratify=y`**: como a classe positiva é minoritária, a estratificação garante que **treino e teste tenham a mesma proporção de classes**. Sem isso, o teste poderia ficar com pouquíssimos casos positivos por sorte.
- O **teste fica "trancado"**: não participa de ajuste de transformações nem da escolha de hiperparâmetros.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Treino: {X_treino.shape[0]} linhas | Teste: {X_teste.shape[0]} linhas")
prop = pd.DataFrame({
    "treino (qtde)": y_treino.value_counts().sort_index(),
    "treino (%)": (y_treino.value_counts(normalize=True).sort_index() * 100).round(1),
    "teste (qtde)": y_teste.value_counts().sort_index(),
    "teste (%)": (y_teste.value_counts(normalize=True).sort_index() * 100).round(1),
})
prop.index = [ROTULOS[i] for i in prop.index]
display(prop)

## 8) Algoritmos e hiperparâmetros

### Algoritmos escolhidos

| Algoritmo | Por que escolhi | O que espero | Sensível à escala? | Hiperparâmetros testados |
|---|---|---|---|---|
| **Baseline** (`DummyClassifier`, sempre a classe majoritária) | Referência mínima | Boa acurácia, **F1 = 0** (nunca acerta a classe positiva) | Não | — |
| **Regressão Logística** | Simples e interpretável; as classes parecem separáveis por uma fronteira linear | Bom desempenho, estável | **Recomendada** | `C`, `class_weight` |
| **Árvore de Decisão** | Regras fáceis de explicar ("se abdômen > X…") | Tende a superajustar se profunda | Não | `max_depth`, `class_weight` |
| **KNN** | Classifica olhando pessoas com medidas parecidas | Competitivo **se padronizado** | **Sim** | `n_neighbors`, `weights` |
| **Random Forest** | Conjunto de árvores: menos variância e superajuste | Bom e estável | Não | `n_estimators`, `class_weight` |
| **SVC (SVM)** | Fronteiras lineares ou curvas (kernel); forte em dados numéricos pequenos | Muito bom com padronização | **Sim** | `kernel`, `C`, `class_weight` |
| **SGDClassifier** | Aprende por gradiente, passo a passo; mostra o efeito da taxa de aprendizagem | Sensível a `eta0` e ao nº de iterações | **Sim** | `eta0`, `max_iter` |

### Como escolho hiperparâmetros sem "espiar" o teste

Uso **validação cruzada estratificada de 5 folds** (`StratifiedKFold(shuffle=True, random_state=42)`) **somente dentro do treino**. A **escolha** é feita pelo **F1 da classe positiva** na validação cruzada. As métricas de teste são mostradas para comparação, mas **não decidem** a escolha. A padronização fica **dentro de `Pipeline`**: em cada fold, o `StandardScaler` aprende média e desvio **só do treino daquele fold** (sem *data leakage*).

> Regra de organização: altero **um hiperparâmetro por vez** e registro tudo numa tabela.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_pos = make_scorer(f1_score, zero_division=0)      # F1 da classe positiva (1)
resultados = []     # tabela de experimentos
modelos = {}        # modelos treinados + previsões no teste


def medir(modelo):
    """F1 em validação cruzada (treino) e métricas no teste."""
    cv_f1 = cross_val_score(modelo, X_treino, y_treino, cv=skf, scoring=f1_pos)
    modelo.fit(X_treino, y_treino)
    prev = modelo.predict(X_teste)
    return {
        "cv_f1": cv_f1.mean(),
        "cv_desvio": cv_f1.std(),
        "acuracia": accuracy_score(y_teste, prev),
        "precisao": precision_score(y_teste, prev, zero_division=0),
        "recall": recall_score(y_teste, prev, zero_division=0),
        "f1": f1_score(y_teste, prev, zero_division=0),
        "prev": prev,
    }


def registrar(familia, hiper, modelo):
    m = medir(modelo)
    resultados.append({"familia": familia, "hiperparametros": hiper,
                       **{k: v for k, v in m.items() if k != "prev"}})
    modelos[(familia, hiper)] = (modelo, m["prev"])


def mostrar(familia):
    tabela = pd.DataFrame(resultados).query("familia == @familia").drop(columns="familia")
    display(tabela.round(3).reset_index(drop=True))


def com_escala(estimador):
    return Pipeline([("escala", StandardScaler()), ("modelo", estimador)])

In [ ]:
# Baseline
registrar("Baseline (maioria)", "strategy='most_frequent'", DummyClassifier(strategy="most_frequent"))
mostrar("Baseline (maioria)")

**Leitura do baseline.** Repare na **acurácia alta com F1 = 0**: o baseline acerta quase todos os negativos e **não encontra nenhum positivo**. É exatamente por isso que a acurácia, sozinha, não serve como critério neste problema.

In [ ]:
# Regressão Logística — experimento 1: força da regularização (C) — com padronização
for c in [0.1, 1, 10]:
    registrar("Regressão Logística", f"C={c} (com escala)",
              com_escala(LogisticRegression(C=c, max_iter=5000, random_state=42)))

# Experimento 2: dar mais peso à classe minoritária (C = 1)
registrar("Regressão Logística", "C=1, class_weight='balanced' (com escala)",
          com_escala(LogisticRegression(C=1, class_weight="balanced", max_iter=5000, random_state=42)))
mostrar("Regressão Logística")

**Leitura.** `C` controla a regularização (menor `C` = modelo mais "contido"). `class_weight='balanced'` faz o erro na classe minoritária **custar mais** no treino, o que costuma **aumentar o recall** (achar mais positivos) ao custo de **menor precisão**.

In [ ]:
# Árvore de Decisão — experimento 1: profundidade máxima
for prof in [2, 3, 5, 10]:
    registrar("Árvore de Decisão", f"max_depth={prof}",
              DecisionTreeClassifier(max_depth=prof, random_state=42))

# Experimento 2: class_weight (max_depth = 3)
registrar("Árvore de Decisão", "max_depth=3, class_weight='balanced'",
          DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42))
mostrar("Árvore de Decisão")

**Leitura.** Árvore muito profunda "decora" o treino; muito rasa não captura o padrão. O melhor ponto costuma estar onde o `cv_f1` é máximo. Compare `cv_f1` com `f1` (teste): uma grande diferença indica superajuste.

In [ ]:
# KNN — experimento 1: número de vizinhos (com padronização)
for k in [3, 5, 9, 15]:
    registrar("KNN", f"n_neighbors={k} (com escala)",
              com_escala(KNeighborsClassifier(n_neighbors=k)))

# Experimento 2: peso dos vizinhos (k = 9)
registrar("KNN", "n_neighbors=9, weights='distance' (com escala)",
          com_escala(KNeighborsClassifier(n_neighbors=9, weights="distance")))
mostrar("KNN")

**Leitura.** Com poucos vizinhos o KNN é sensível a ruído; com muitos, "suaviza" demais e a classe minoritária tende a ser **engolida pela maioria**. O KNN entra sempre **com padronização** (seção 9).

In [ ]:
# Random Forest — experimento 1: número de árvores
for n in [50, 100, 200]:
    registrar("Random Forest", f"n_estimators={n}",
              RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1))

# Experimento 2: class_weight (n_estimators = 200)
registrar("Random Forest", "n_estimators=200, class_weight='balanced'",
          RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1))
mostrar("Random Forest")

**Leitura.** Mais árvores estabilizam o resultado, com ganho decrescente. `class_weight='balanced'` também pode ser aplicado aqui para dar mais atenção à classe minoritária.

In [ ]:
# SVC — experimento 1: kernel e C (com padronização)
registrar("SVC", "kernel='linear', C=1 (com escala)",
          com_escala(SVC(kernel="linear", C=1.0)))
for c in [0.1, 1, 10]:
    registrar("SVC", f"kernel='rbf', C={c} (com escala)",
              com_escala(SVC(kernel="rbf", C=c, gamma="scale")))

# Experimento 2: class_weight (rbf, C = 1)
registrar("SVC", "kernel='rbf', C=1, class_weight='balanced' (com escala)",
          com_escala(SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")))
mostrar("SVC")

**Leitura.** O kernel `linear` traça uma fronteira reta; o `rbf` permite fronteira curva. `C` pequeno aceita mais erros no treino (modelo mais simples); `C` grande tenta acertar tudo (risco de superajuste). Compare linear × rbf nas colunas `cv_f1` e `f1`.

In [ ]:
# SGDClassifier — experimento 1: taxa de aprendizagem (eta0), com padronização
for eta in [0.01, 0.001, 0.0001]:
    registrar("SGD", f"eta0={eta}, max_iter=1000 (com escala)",
              com_escala(SGDClassifier(loss="log_loss", learning_rate="constant", eta0=eta,
                                       max_iter=1000, tol=None, random_state=42)))

# Experimento 2: número de iterações (eta0 = 0.001)
for it in [500, 3000]:
    registrar("SGD", f"eta0=0.001, max_iter={it} (com escala)",
              com_escala(SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.001,
                                       max_iter=it, tol=None, random_state=42)))
mostrar("SGD")

**Leitura.** O SGD aprende em pequenos passos. Com `eta0` muito baixo e poucas iterações, o modelo **não converge** (subajuste); com `eta0` alto, o treino pode ficar **instável**. Veja se o `cv_f1` melhora ao aumentar `max_iter` e se o `cv_desvio` indica oscilação entre folds.

## 9) Necessidade de normalização

- **KNN** usa **distância euclidiana**: variáveis de maior magnitude (peso ≈ 80 kg, abdômen ≈ 92 cm) **dominam** a distância e as pequenas (punho, tornozelo) quase não influenciam.
- **SVC, Regressão Logística e SGD** também dependem da escala (kernel/otimização por gradiente/regularização).
- **Árvores e Random Forest** decidem por **limiares em uma variável por vez**; a escala não muda a ordem dos valores. **Não dependem de escala**.

Para provar com dados, comparo **com e sem** padronização (o scaler continua dentro do Pipeline, aprendendo só do treino):

In [ ]:
experimentos = [
    ("KNN (k=9)", KNeighborsClassifier(n_neighbors=9)),
    ("SVC (rbf, C=1)", SVC(kernel="rbf", C=1.0, gamma="scale")),
    ("Regressão Logística (C=1)", LogisticRegression(C=1, max_iter=5000, random_state=42)),
    ("SGD (eta0=0.001)", SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.001,
                                       max_iter=1000, tol=None, random_state=42)),
    ("Árvore (max_depth=3)", DecisionTreeClassifier(max_depth=3, random_state=42)),
]
from sklearn.base import clone

comparacao = []
for nome, est in experimentos:
    for rotulo, mod in [("com StandardScaler", com_escala(clone(est))), ("sem escala", clone(est))]:
        m = medir(mod)
        comparacao.append({"modelo": nome, "versão": rotulo, "cv_f1": m["cv_f1"],
                           "acuracia_teste": m["acuracia"], "f1_teste": m["f1"]})

df_norm = pd.DataFrame(comparacao)
display(df_norm.round(3))

for nome in ["KNN (k=9)", "SVC (rbf, C=1)"]:
    t = df_norm[df_norm["modelo"] == nome].set_index("versão")["cv_f1"]
    print(f"Leitura automática ({nome}): F1 em validação = {t['com StandardScaler']:.3f} com escala "
          f"vs {t['sem escala']:.3f} sem escala.")

**Interpretação.** Para os modelos sensíveis à escala (KNN, SVC, Regressão Logística, SGD), compare as duas linhas de cada modelo: a diferença mostra o efeito de não padronizar. A **Árvore** deve dar **resultados iguais** com e sem escala, o que confirma que ela não depende dela. Em amostra pequena o número nem sempre melhora com a padronização, mas ela torna o resultado **independente das unidades** (kg ou lb, cm ou pol). Por isso, na comparação final, os modelos sensíveis à escala entram **sempre padronizados**.

## 10) Comparação por métricas

### O que cada métrica diz (classe positiva = gordura elevada)

| Métrica | Interpretação | Por que importa aqui |
|---|---|---|
| **Acurácia** | % geral de acertos | Pode enganar com classes desbalanceadas (o baseline mostra isso). |
| **Precisão** | Quando o modelo diz "gordura elevada", com que frequência acerta? | Evita **falsos alarmes** (alertar quem não precisa). |
| **Recall** | Dos que realmente têm gordura elevada, quantos o modelo encontrou? | Evita **deixar passar** quem precisa de atenção. |
| **F1-score** | Equilíbrio entre precisão e recall | **Métrica principal**: resume os dois tipos de erro numa classe minoritária. |
| **Matriz de confusão** | Acertos e erros por classe | Mostra **onde** o modelo erra (falsos positivos × falsos negativos). |

Não escolho "o maior número": escolho pelo **F1 da classe positiva** (validado por validação cruzada) e **confiro** com recall, precisão e a matriz de confusão.

In [ ]:
df_res = pd.DataFrame(resultados)
tabela_final = df_res.sort_values("cv_f1", ascending=False).reset_index(drop=True)
display(tabela_final.round(3))

# Melhor configuração por família: validação cruzada x teste
melhores = df_res.loc[df_res.groupby("familia")["cv_f1"].idxmax()].copy()
grafico = melhores.melt(id_vars="familia", value_vars=["cv_f1", "f1"],
                        var_name="Avaliação", value_name="F1")
grafico["Avaliação"] = grafico["Avaliação"].map({"cv_f1": "Validação cruzada (treino)", "f1": "Teste"})

plt.figure(figsize=(9, 5))
sns.barplot(data=grafico, y="familia", x="F1", hue="Avaliação",
            order=melhores.sort_values("cv_f1", ascending=False)["familia"])
plt.title("Melhor configuração de cada algoritmo: F1 da classe positiva (maior é melhor)")
plt.xlabel("F1-score (classe: gordura elevada)")
plt.ylabel("Algoritmo")
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Matrizes de confusão (teste) da melhor configuração de cada algoritmo
fam = melhores.sort_values("cv_f1", ascending=False)
n = len(fam); colunas = 4; linhas = int(np.ceil(n / colunas))
fig, axes = plt.subplots(linhas, colunas, figsize=(4.2 * colunas, 3.8 * linhas))
axes = np.array(axes).ravel()
for ax, (_, r) in zip(axes, fam.iterrows()):
    _, prev = modelos[(r["familia"], r["hiperparametros"])]
    cm = confusion_matrix(y_teste, prev, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=[ROTULOS_CURTOS[0], ROTULOS_CURTOS[1]],
                yticklabels=[ROTULOS_CURTOS[0], ROTULOS_CURTOS[1]])
    ax.set(title=f"{r['familia']}\nF1 = {r['f1']:.2f} | recall = {r['recall']:.2f}",
           xlabel="Classe prevista", ylabel="Classe real")
for ax in axes[n:]:
    ax.axis("off")
plt.suptitle("Matrizes de confusão no teste (linhas = real, colunas = previsto)", y=1.02)
plt.tight_layout()
plt.show()

**Como ler.** A diagonal principal são os **acertos**. O canto inferior esquerdo são os **falsos negativos** (tinha gordura elevada e o modelo disse que não); o superior direito são os **falsos positivos** (o modelo alertou sem necessidade). O teste tem **poucos casos positivos**, então cada erro pesa bastante no F1 e as diferenças pequenas entre modelos não são conclusivas.

In [ ]:
# Escolha do melhor modelo: maior F1 na VALIDAÇÃO CRUZADA (sem olhar o teste)
candidatos = df_res[df_res["familia"] != "Baseline (maioria)"]
melhor = df_res.loc[candidatos["cv_f1"].idxmax()]
base = df_res[df_res["familia"] == "Baseline (maioria)"].iloc[0]
modelo_final, prev_final = modelos[(melhor["familia"], melhor["hiperparametros"])]

display(Markdown(f"""
### Modelo escolhido: **{melhor['familia']}** com `{melhor['hiperparametros']}`

| | CV F1 (treino) | Acurácia (teste) | Precisão (teste) | Recall (teste) | F1 (teste) |
|---|---|---|---|---|---|
| **{melhor['familia']}** | {melhor['cv_f1']:.3f} ± {melhor['cv_desvio']:.3f} | {melhor['acuracia']:.3f} | {melhor['precisao']:.3f} | {melhor['recall']:.3f} | {melhor['f1']:.3f} |
| Baseline (sempre a maioria) | {base['cv_f1']:.3f} | {base['acuracia']:.3f} | {base['precisao']:.3f} | {base['recall']:.3f} | {base['f1']:.3f} |
"""))

top_acc = candidatos.loc[candidatos["acuracia"].idxmax()]
top_f1 = candidatos.loc[candidatos["f1"].idxmax()]
print("O modelo de maior acurácia no teste é o mesmo de maior F1 no teste?",
      "SIM" if (top_acc["familia"], top_acc["hiperparametros"]) == (top_f1["familia"], top_f1["hiperparametros"]) else "NÃO")
print(f"  maior acurácia: {top_acc['familia']} ({top_acc['hiperparametros']}) = {top_acc['acuracia']:.3f}")
print(f"  maior F1:       {top_f1['familia']} ({top_f1['hiperparametros']}) = {top_f1['f1']:.3f}")

### Por que esse modelo? (justificativa)

1. Foi o de **maior F1 na validação cruzada**, escolha feita **sem usar o teste**.
2. O F1 no teste, comparado com o da validação, indica se o modelo **generaliza** (números próximos sugerem que não houve superajuste).
3. O ganho sobre o baseline mostra que as medidas corporais **realmente carregam informação** sobre a classe, algo que a acurácia do baseline esconde.
4. Se dois modelos tiverem desempenho muito próximo (diferença menor que o desvio da validação cruzada), prefira o **mais simples e explicável**.

In [ ]:
# Análise dos erros do melhor modelo: onde ele erra?
tn, fp, fn, tp = confusion_matrix(y_teste, prev_final, labels=[0, 1]).ravel()
print(f"Verdadeiros negativos: {tn} | Falsos positivos: {fp} | Falsos negativos: {fn} | Verdadeiros positivos: {tp}")

erros = pd.DataFrame({"gordura_real_%": y_pct.loc[X_teste.index],
                      "classe_real": y_teste, "classe_prevista": prev_final}, index=X_teste.index)
erros = erros[erros["classe_real"] != erros["classe_prevista"]]
display(erros.round(1))
if len(erros):
    dist = (erros["gordura_real_%"] - LIMIAR).abs()
    print(f"Distância média dos erros até o limiar de {LIMIAR:.0f}%: {dist.mean():.1f} p.p.")

**Interpretação.** Se os erros estão **perto do limiar de 25 %**, o modelo não está "errando feio": ele erra justamente onde a classe é naturalmente ambígua (alguém com 24,5 % e outro com 25,5 % são fisicamente quase iguais). Esse é um limite do problema, e não só do modelo.

In [ ]:
# Importância por permutação: quanto o F1 piora ao embaralhar cada variável (conjunto de teste)
pi = permutation_importance(modelo_final, X_teste, y_teste, n_repeats=30, random_state=42,
                            scoring=f1_pos)
imp = pd.Series(pi.importances_mean, index=X_teste.columns).sort_values()

plt.figure(figsize=(8, 5))
imp.plot.barh(color="steelblue")
plt.title("Importância das variáveis no melhor modelo (permutação)")
plt.xlabel("Queda do F1 ao embaralhar a variável")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()
print("Variáveis mais importantes:", ", ".join(imp.sort_values(ascending=False).index[:3]))

**Interpretação.** Barras maiores indicam variáveis das quais o modelo mais depende. Variáveis com barra próxima de zero muitas vezes **não são inúteis**: a informação delas já está em outra feature correlacionada (multicolinearidade).

## 11) Conclusão final

In [ ]:
top_imp = imp.sort_values(ascending=False).index[:3]
lista_imp = ", ".join(f"`{c}`" for c in top_imp)
melhor_por_familia = melhores.set_index("familia")["cv_f1"].drop("Baseline (maioria)").sort_values(ascending=False)
ranking = " | ".join(f"{k} = {v:.3f}" for k, v in melhor_por_familia.items())

display(Markdown(f"""
### Síntese dos resultados (gerada a partir desta execução)

- **Dados:** {len(df_bruto)} homens originais; após a limpeza ficaram **{len(X)}** (X: {X.shape[1]} features).
- **Classes:** {int(y.sum())} com gordura elevada (≥ {LIMIAR:.0f}%) e {int((1 - y).sum())} abaixo ({y.mean():.1%} positivos). **Desbalanceado.**
- **Modelo escolhido:** {melhor['familia']} — `{melhor['hiperparametros']}`.
- **Desempenho no teste:** acurácia = {melhor['acuracia']:.3f}, precisão = {melhor['precisao']:.3f}, recall = {melhor['recall']:.3f}, F1 = {melhor['f1']:.3f}.
- **Baseline (sempre a maioria):** acurácia = {base['acuracia']:.3f}, mas F1 = {base['f1']:.3f}.
- **Variáveis mais importantes (permutação):** {lista_imp}.
- **Melhor F1 de validação cruzada por algoritmo:** {ranking}.
"""))

### O que foi aprendido

**Sobre o dataset e as classes.** Com o limiar de 25 %, a classe de gordura elevada é **minoritária**. As medidas de tronco (abdômen, peso, peito e quadril) separam bem as classes, enquanto altura e punho contribuem pouco. Muitas features são redundantes entre si.

**Problemas de qualidade e pré-processamento.**
- Removi do `X` todas as colunas **derivadas do % de gordura** (densidade, Brozek, massa magra), pois entregariam a resposta (*target leakage*).
- Corrigi uma **altura impossível** (erro de digitação documentado) e removi poucas linhas com **target não confiável**, porque o rótulo delas também seria duvidoso.
- Converti unidades por regra fixa e **não removi outliers reais** (eles pertencem justamente à classe que queremos detectar).
- A padronização ficou **dentro de Pipeline**, ajustada só no treino (e por fold na validação cruzada), e a divisão usou **`stratify=y`**.

**Gráficos que mais ajudaram.** O de balanceamento (mostrou por que a acurácia engana) e o scatter abdômen × peso (mostrou a separação das classes e a zona de transição).

**Modelos e hiperparâmetros.** Testei Regressão Logística, Árvore, KNN, Random Forest, SVC e SGD, contra um baseline. Nas árvores, a profundidade controla o superajuste; no KNN e no SVC, a padronização e `k`/`C`/kernel fizeram diferença; `class_weight='balanced'` tende a aumentar o recall; no SGD, a taxa de aprendizagem e o número de iterações decidem se o treino converge. A escolha foi por **F1 em validação cruzada estratificada**, e o teste só reportou o desempenho final.

**Normalização.** Importante para KNN, SVC, Regressão Logística e SGD; irrelevante para Árvore e Random Forest, como demonstrado na seção 9.

**Métricas que sustentam a escolha.** **F1 da classe positiva** (principal), apoiado por **recall**, **precisão** e **matriz de confusão**. A acurácia foi usada apenas como contraste com o baseline.

### Limitações e próximos passos
- **Amostra pequena** (~250 homens) e **apenas homens**; o **teste tem poucos casos positivos**, então o F1 varia bastante conforme a divisão e as diferenças pequenas entre modelos não são conclusivas. O ideal seria repetir a validação cruzada para o conjunto todo.
- O **limiar de 25 % é uma escolha de domínio**; outro limiar mudaria o balanceamento e os resultados. Os erros se concentram perto dele.
- A classe é derivada de um valor contínuo: parte da informação (o valor exato) é descartada. A Parte 1 (regressão) usa essa informação completa.
- Próximos passos: `GridSearchCV`, ajuste do limiar de decisão, `predict_proba` e curva ROC/PR, *gradient boosting* e dados mais diversos.
- **Uso exclusivamente educacional**, sem finalidade de diagnóstico.

### Referências
- Johnson, R. W. (1996). *Fitting Percentage of Body Fat to Simple Body Measurements*. Journal of Statistics Education, 4(1). Dados: https://jse.amstat.org/datasets/fat.txt
- Penrose, K., Nelson, A., Fisher, A. (1985). *Generalized Body Composition Prediction Equation for Men Using Simple Measurement Techniques*. Medicine and Science in Sports and Exercise, 17(2), 189.
- Documentação do scikit-learn: https://scikit-learn.org/stable/

### Checklist da Questão 2

| Item | Status |
|---|---|
| Dataset real e público, com link da fonte | ✅ |
| Dataset não utilizado em aula | ☐ confirmar com o material da disciplina |
| Problema, classe prevista e tipo (binária) explicados | ✅ |
| Dados em DataFrame, X e y separados | ✅ |
| Amostras, features, tipos, estatísticas, ausentes e duplicidades verificados | ✅ |
| Balanceamento das classes verificado e discutido | ✅ |
| Pré-processamento justificado | ✅ |
| Data leakage evitado (colunas derivadas do target removidas; scaler dentro do Pipeline) | ✅ |
| Pelo menos 2 gráficos interpretados | ✅ (3 gráficos + matrizes de confusão) |
| Divisão treino/teste com `random_state=42` e `stratify=y` | ✅ |
| Pelo menos 2 algoritmos e 3 configurações de hiperparâmetros | ✅ (6 algoritmos, várias configurações) |
| Necessidade de normalização explicada e testada | ✅ |
| Métricas adequadas comparadas (acurácia, precisão, recall, F1, matriz de confusão) | ✅ |
| Melhor modelo justificado | ✅ |
| Conclusão final | ✅ |
| Notebook executa do início ao fim sem erros | ☐ rodar *Ambiente de execução → Reiniciar e executar tudo* |